# DocTrust-VLM — one-notebook MVP

This notebook orchestrates the complete small experiment while keeping the reusable logic in `src/doctrust/`.

**What it tests**

- Clean document: preserve the answer.
- JPEG/blur: preserve the answer under nuisance degradation.
- Matched distractor occlusion: preserve the answer when irrelevant content is covered.
- Evidence occlusion: output `UNANSWERABLE` when answer evidence is removed.

**Important:** no model was downloaded or run when this notebook was created. Cells marked **GPU / DOWNLOAD** are the expensive boundary.


## 0. How to use this notebook

Run cells from top to bottom. Start with exactly one document. Inspect every image variant before loading the model.

Use the existing interpreter:

`/home/tharun/Projects/hf-course/hf.venv/bin/python`

In VS Code, select that interpreter as the notebook kernel. For browser Jupyter, install `requirements-notebook.txt` first.


In [ ]:
from pathlib import Path
import os
import sys

# Works whether Jupyter opens at the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)


## 1. Install/check dependencies

Run the `%pip` line once if packages are missing. `%pip` installs into the active notebook kernel, unlike an arbitrary shell `pip`. Restart the kernel after installation if Jupyter requests it.


In [ ]:
# Uncomment and run once if needed:
# %pip install -r requirements.txt

# This script reports packages and GPU state without downloading a model.
%run scripts/check_environment.py


## 2. Describe one document-question pair

A local smoke-test item has already been downloaded from `nielsr/docvqa_1200_examples` (DocVQA Task 1, train row 1 / `train_3`). Its image is deliberately gitignored; provenance is recorded in `data/manifests/docvqa-train-3-provenance.json`.

`EVIDENCE_BOX` is normalized `[x1, y1, x2, y2]`. It tightly surrounds the visible answer text `T.F. Riehl`.


In [ ]:
IMAGE_PATH = Path("data/raw/docvqa-train-3.jpg")
SAMPLE_ID = "docvqa-train-3"
QUESTION = "Who is in cc in this letter?"
ANSWERS = ["T.F. Riehl"]
EVIDENCE_BOX = [0.235, 0.220, 0.315, 0.245]

assert IMAGE_PATH.exists(), f"Add your image first: {IMAGE_PATH}"
assert len(EVIDENCE_BOX) == 4
assert all(0 <= value <= 1 for value in EVIDENCE_BOX)
print("Input configured:", SAMPLE_ID)


## 3. Preview and audit the evidence box

The amber rectangle must cover the answer—not the question label, an unrelated value or most of the page. A wrong box invalidates the experiment.


In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display
from doctrust.corruptions import box_to_pixels

image = Image.open(IMAGE_PATH).convert("RGB")
preview = image.copy()
box_pixels = box_to_pixels(tuple(EVIDENCE_BOX), preview.size)
ImageDraw.Draw(preview).rectangle(box_pixels, outline=(255, 170, 0), width=5)
print("Image size:", image.size, "Evidence box in pixels:", box_pixels)
display(preview)


## 4. Write the source manifest

JSONL makes each question/image pair explicit and reproducible. The notebook calls the same repository function used by the command-line workflow.


In [ ]:
from doctrust.io import write_jsonl

source_manifest = Path("data/manifests/source.jsonl")
write_jsonl(source_manifest, [{
    "id": SAMPLE_ID,
    "image_path": str(IMAGE_PATH),
    "question": QUESTION,
    "answers": ANSWERS,
    "evidence_box": EVIDENCE_BOX,
}])
print("Wrote:", source_manifest)


## 5. Generate controlled variants

This is image preprocessing only; it does not load a VLM. The same answer-bearing region determines both evidence and matched distractor masks.


In [ ]:
from doctrust.prepare import prepare

prepared_manifest = prepare("configs/mvp.yaml")
print("Prepared:", prepared_manifest)


## 6. Inspect every generated image

Do not continue unless:

- JPEG and blur leave the answer human-readable;
- distractor occlusion does not touch answer evidence;
- evidence occlusion fully removes the answer.


In [ ]:
from doctrust.io import read_jsonl

prepared_rows = read_jsonl(prepared_manifest)
for row in prepared_rows:
    variant_image = Image.open(row["image_path"]).convert("RGB")
    print(row["variant"], "→ expected:", row["expected_behavior"])
    display(variant_image.resize((min(900, variant_image.width), int(variant_image.height * min(900, variant_image.width) / variant_image.width))))


## 7. **GPU / DOWNLOAD boundary** — load Granite Vision

Running the next cell downloads model files on first use and allocates GPU/CPU memory. The configuration requests 4-bit NF4 loading, float16 computation and batch size one.

If this cell raises CUDA OOM, restart the kernel, close GPU-heavy applications and verify free VRAM with `nvidia-smi`. Do not continue with partially loaded state.


In [ ]:
from doctrust.config import load_config
from doctrust.modeling import DocumentVLM

config = load_config("configs/mvp.yaml")
print("Loading:", config["model"]["model_id"])
model = DocumentVLM(config["model"])
print("Model loaded.")


## 8. Run deterministic inference

`do_sample=False` removes sampling variability. Each prediction is appended immediately, so an interrupted run can resume. Delete `results/predictions.jsonl` only when you intentionally want a fresh run.


In [ ]:
from doctrust.run import run

predictions_path = run("configs/mvp.yaml")
predictions = read_jsonl(predictions_path)
print("Predictions written:", predictions_path)
for row in predictions:
    print(f"{row['variant']:24s} | {row['prediction']!r} | {row['latency_seconds']:.2f}s | {row['peak_vram_mb']} MB")


## 9. Compute metrics

- ANLS evaluates answer-preserving variants.
- Abstention rate evaluates evidence removal.
- False-answer rate is the dangerous case: the model supplies an answer after visible evidence was removed.


In [ ]:
import json
from doctrust.evaluate import evaluate

metrics = evaluate(predictions_path)
metrics_path = Path("results/metrics.json")
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metrics, indent=2))


## 10. Interpret before scaling

For one example, the output is a pipeline check—not research evidence.

Before expanding to 20–50 examples, answer these:

1. Was the clean answer correct?
2. Did nuisance degradation preserve a human-readable answer?
3. Was distractor damage genuinely irrelevant?
4. Was evidence removal complete?
5. Did the model abstain, hallucinate or copy a stale answer?
6. Are raw outputs and resource measurements recorded?

Only after this passes should you add more examples. Read `docs/EXPERIMENT_DESIGN.md` for claim boundaries and the next-stage semantic-counterfactual study.
